<a href="https://colab.research.google.com/github/Hashim123132/flyrank-ml-internship/blob/main/work/notebooks/w05_model.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-08: Capstone Modeling Lane

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Hashim123132/flyrank-ml-internship/blob/main/work/notebooks/w05_model.ipynb?flush_cache=true)

Lane 1 (Ranking Signal Analysis) on the starter slice (30,000 rows, one row per content item). The learned model is trained, then compared against the Week-4 baseline rule on the same data, the same split, and the same metric. No warehouse needed.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the `training-honest-models` skill + `flyrank/flyrank-data`.

## 1. Method choice and why

The lane is ranking signal analysis: per page, the question is yes/no ("is this page at decline risk?"), and the label is observed in the data (`down`, from `trend_direction`, evaluation only, never a feature). That maps to the toolkit row: start with **Logistic Regression**, then **Random Forest**.

Logistic first because it is readable: its coefficients say how much each observed signal moves the risk, in the same units the Week-4 rule already uses (position, CTR, impressions). Random forest second because the Week-4 signal audit found position-tier by CTR interactions (median CTR steps 0.24 > 0.20 > 0.17 > 0.09 > 0.00) that a linear model may miss. Keep it only if it actually beats the logistic model.

Both are compared to the Week-4 baseline rule on the same slice, the same split, and the same metric (precision@K), as the honest-model skill requires.

In [6]:
import pandas as pd, numpy as np
import sklearn
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.inspection import permutation_importance
from sklearn.metrics import roc_auc_score

SEED = 42
print("sklearn", sklearn.__version__, "| numpy", np.__version__, "| pandas", pd.__version__)

df = pd.read_csv("content_refresh_anonymized.csv")
df["down"] = (df.trend_direction == "down").astype(int)   # eval only, never a feature
print("rows:", len(df), "| clients:", df.client_id.nunique())

sklearn 1.6.1 | numpy 2.0.2 | pandas 2.2.2
rows: 30000 | clients: 32


## 2. Split design

**Client holdout** using the repo's own split: a seed-42 permutation of the 32 clients, the first 20% become the test set. Pages of one client share SERP and tracking patterns (the Week-4 top-20 itself flagged two top picks on one client as a possible client-level pattern), so a random row split would leak those patterns into the test set and flatter the model. Same split for the baseline and both models.

One honest detail: the rule's tier-median CTR is recomputed from **train rows only** and then applied to test. The Week-4 notebook computed it on the full slice; this run re-derives it so the baseline never peeks at test CTRs.

In [7]:
# Client holdout, the repo's own split: default_rng(42).permutation(clients), first 20% = test.
v = df[(df.impressions_90d >= 500) & (df.avg_position > 0)].copy()
clients = v.client_id.drop_duplicates().to_numpy()
shuffled = np.random.default_rng(SEED).permutation(clients)
n_test = max(1, int(round(len(shuffled) * 0.2)))
test_clients = set(shuffled[:n_test])
is_test = v.client_id.isin(test_clients)
train, test = v[~is_test], v[is_test]

print("clients:", len(clients), "| held out:", len(test_clients))
print("train rows:", len(train), "| test rows:", len(test))
print("test base rate (down):", round(test.down.mean(), 3))

clients: 28 | held out: 6
train rows: 12939 | test rows: 3787
test base rate (down): 0.633


## 3. Train + compare vs my baseline

Features are **observed signals only**: no `trend_*`, no label, no IDs, no provider or model columns. Missing values keep the data skill's rule: add `has_`-flags (missingness follows `content_type`) and fill with **train** medians, so test rows are filled without peeking.

The baseline rule is rebuilt inside this run: score = `(avg_position <= 20) * max(0, tier median CTR - page CTR) * log1p(impressions_90d)`, tier medians from train only. Same split, same metric (precision@K), same notebook run. For continuity: the Week-4 card numbers (P@10 0.70 / P@20 0.70 / P@50 0.68, base rate 0.596) were measured on the full slice with full-slice tier medians; they are not comparable to test-set numbers, which is exactly why the baseline is re-derived here.

In [8]:
# Features: observed signals only.
LOG_COLS = ["impressions_90d", "clicks_90d", "sessions_90d", "days_with_impressions",
            "word_count", "search_volume"]
RAW_COLS = ["avg_position", "ctr", "content_age_days", "days_since_last_update",
            "engagement_rate", "scroll_rate", "ai_traffic_pct"]

def make_x(d):
    x = pd.DataFrame(index=d.index)
    for c in LOG_COLS:
        x["log_" + c] = np.log1p(d[c])
    for c in RAW_COLS:
        x[c] = d[c]
    x["has_word_count"] = d.word_count.notna().astype(int)
    x["has_keyword_data"] = d.search_volume.notna().astype(int)
    for t in sorted(d.position_tier.unique()):
        x["tier_" + t] = (d.position_tier == t).astype(int)
    return x

Xtr, Xte = make_x(train), make_x(test)
med = Xtr.median()
Xtr, Xte = Xtr.fillna(med), Xte.fillna(med)
ytr, yte = train.down.values, test.down.values

# Week-4 baseline rule, rebuilt in this run: tier medians from TRAIN only.
tier_ctr = train.groupby("position_tier").ctr.median()

def baseline_score(d):
    return ((d.avg_position <= 20).astype(int)
            * ((d.position_tier.map(tier_ctr)) - d.ctr).clip(lower=0)
            * np.log1p(d.impressions_90d))

s_base = baseline_score(test)

def p_at_k(scores, k):
    order = np.argsort(-np.asarray(scores))
    return yte[order[:k]].mean()

rows = {"baseline rule": s_base}

lr = make_pipeline(StandardScaler(),
                   LogisticRegression(max_iter=5000, random_state=SEED)).fit(Xtr, ytr)
rows["logistic regression"] = lr.predict_proba(Xte)[:, 1]

rf = RandomForestClassifier(n_estimators=300, random_state=SEED, n_jobs=4).fit(Xtr, ytr)
rows["random forest"] = rf.predict_proba(Xte)[:, 1]

table = pd.DataFrame(
    {"P@10": [p_at_k(s, 10) for s in rows.values()],
     "P@20": [p_at_k(s, 20) for s in rows.values()],
     "P@50": [p_at_k(s, 50) for s in rows.values()],
     "AUC":  [roc_auc_score(yte, s) for s in rows.values()]},
    index=rows.keys()).round(3)

print("Comparison on the held-out test set: same slice, same split, same metrics")
print(table)
print("base rate (random pick on test):", round(yte.mean(), 3))

Comparison on the held-out test set: same slice, same split, same metrics
                     P@10  P@20  P@50    AUC
baseline rule         0.8  0.85  0.88  0.572
logistic regression   0.9  0.95  0.88  0.638
random forest         0.8  0.85  0.84  0.589
base rate (random pick on test): 0.633


## 4. Errors and interpretation

**Read the size first.** The held-out test is 3,787 rows, so P@10 covers 38 rows and P@20 covers 76; the table is directional, not a guarantee.

**What it leans on.** Scaled logistic coefficients and permutation importance point the same way: fewer clicks, worse rank (higher `avg_position` number), younger content, and more impressions all raise risk. Each is plausible: decline shows up on visible pages that still get demand but are losing clicks; pages already buried at rank 30+ have little left to lose; young pages decay more than old ones (Week-4 Signal A). No feature is suspiciously perfect (the largest permutation drop is 0.035 in AUC), so nothing looks like leakage; the check below asserts the feature set is disjoint from label sources and IDs.

**Where it is most wrong.** The errors cluster at the most visible ranks: 4 of the 24 `top_3` picks and 2 of the 10 `page_1` picks in the top 50 are not currently down. At the other end, the `deep` tier (rank 25+, n=51 in test) the model is at chance: AUC 0.39. Deep pages sit at CTR 0.00 on tiny impression counts, so every signal that separates decline elsewhere has already bottomed out.

**Why it beats the rule.** The rule never looks past `avg_position <= 20`, so its test top-50 is all `page_1` and `top_3`. The model reads the same gap idea (visible page, weak CTR against demand) but without the hard gate, so it reaches 12 `page_3_5` rows in the top 50 with zero false positives.

**Three wrong cases, and why they are hard.**

1. `content_4a6607efcb46` (top_3, position 2.2, CTR 0.01, 128k impressions, predicted 0.90, not currently down). It is a near-clone of the model's 19 correct top-20 picks; only the label differs. Either the signal set cannot separate it, or this page is a tracking or intent artifact rather than real decay.
2. `content_186489efe6f4` (top_3, position 2.3, CTR 0.06, 3 clicks, not down). That CTR is built on 3 clicks; one click either way flips the rate. Small-count CTR noise is a hard limit for any model fed rates.
3. The deep tier: `content_bfebd3351a9f` (position 54.7, 0 clicks, not down) and `content_6a813f186643` (position 57.0, 0 clicks, down) score almost the same. At rank 50+ with zero clicks there is no signal left to sort on; this is the least reliable part of the queue.

**What this means for the queue.** The learned model is a better ranker at the top (P@10 0.90 vs 0.80, P@20 0.95 vs 0.85), ties the rule at P@50 (0.88), and outputs probabilities, so a queue row can carry a confidence and the same three reasons (demand, position, CTR gap). The Week-4 caveat still holds: the label is a proxy, and a visible page can lose clicks for reasons outside the page (SERP changes, intent mismatch) that no model trained on these signals can see. The random forest did not earn its place (0.589 AUC vs 0.638), so the adopted candidate is the logistic model.

In [9]:
# What the model leans on: scaled LR coefficients (readable) + permutation importance on test.
coefs = sorted(zip(Xtr.columns, lr.named_steps["logisticregression"].coef_[0]),
               key=lambda t: -abs(t[1]))
print("LR top coefficients (scaled; positive = higher decline risk):")
for f, w in coefs[:6]:
    print(f"  {f:26s} {w:+.3f}")

perm = permutation_importance(rf, Xte, yte, n_repeats=5, random_state=SEED,
                              scoring="roc_auc", n_jobs=4)
print("\nRF permutation importance (AUC drop when the column is shuffled):")
for f, m in sorted(zip(Xte.columns, perm.importances_mean), key=lambda t: -t[1])[:6]:
    print(f"  {f:26s} {m:+.4f}")

# Where the model is wrong: top-50 by tier, AUC per tier, concrete cases.
p_lr = rows["logistic regression"]
top50 = test.iloc[np.argsort(-p_lr)[:50]].copy()
print("\nLR top-50 picks by tier (n / not down):")
print(top50.groupby("position_tier").agg(n=("down", "size"),
      not_down=("down", lambda s: (s == 0).sum())).to_string())

print("\nLR AUC by position tier (held-out test, tiers with n >= 30):")
for t in sorted(test.position_tier.unique()):
    m = (test.position_tier == t).values
    if m.sum() >= 30:
        print(f"  {t:9s} n={m.sum():5d} base={test.down[m].mean():.2f} "
              f"AUC={roc_auc_score(yte[m], p_lr[m]):.3f}")

top50_fp = top50[top50.down == 0]
print("\nFalse positives in the LR top-50 (model says decline, not currently down):")
print(top50_fp[["content_id", "position_tier", "avg_position", "ctr",
                "impressions_90d", "clicks_90d", "down"]].head(8).to_string(index=False))

dm = (test.position_tier == "deep").values
deep3 = test[dm].iloc[np.argsort(-p_lr[dm])[:3]]
print("\ndeep-tier top-3 by the model (at-chance tier):")
print(deep3[["content_id", "avg_position", "ctr", "impressions_90d", "down"]].to_string(index=False))

# Leakage check: no label-derived columns, no product flags, no IDs among features.
banned = {"trend_direction", "trend_pct", "is_declining_label"}
print("\nfeatures disjoint from label sources:", set(Xtr.columns).isdisjoint(banned))
print("features disjoint from IDs:", set(Xtr.columns).isdisjoint({"content_id", "client_id"}))

LR top coefficients (scaled; positive = higher decline risk):
  log_clicks_90d             -0.677
  avg_position               -0.489
  content_age_days           -0.321
  log_impressions_90d        +0.320
  scroll_rate                +0.265
  log_days_with_impressions  +0.249

RF permutation importance (AUC drop when the column is shuffled):
  log_clicks_90d             +0.0352
  ctr                        +0.0254
  scroll_rate                +0.0139
  log_days_with_impressions  +0.0102
  log_impressions_90d        +0.0075
  avg_position               +0.0064

LR top-50 picks by tier (n / not down):
                n  not_down
position_tier              
page_1         10         2
page_3_5       12         0
striking        4         0
top_3          24         4

LR AUC by position tier (held-out test, tiers with n >= 30):
  deep      n=   51 base=0.57 AUC=0.392
  page_1    n=  720 base=0.64 AUC=0.669
  page_3_5  n= 1760 base=0.64 AUC=0.627
  striking  n= 1220 base=0.62 AUC=0.638
  

## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled, markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/`, then submit your repo URL on the card. Done.